# SleepSense - Training + Export untuk Flask API
**Team CC26-PSU230 | Coding Camp 2026 DBS Foundation**

### Checklist Coverage
**Main Quest:**
- TF Functional API, Custom Layer, Custom Loss, Custom Callback
- Simpan .keras + SavedModel
- Inference script

**Side Quest:**
- tf.GradientTape custom loop
- TensorBoard logging
- Accuracy >= 85%
- Flask REST API + Gemini (di app/main.py)

In [ ]:
# Cell 1 - Install
!pip install -q scikit-learn pandas numpy matplotlib tensorboard

In [ ]:
# Cell 2 - Upload Dataset
from google.colab import files
import os
print("Upload sleepsense_model_ready.csv")
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print("File:", DATA_PATH)

In [ ]:
# Cell 3 - Import & Setup
import os, json, warnings, datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = "/content/sleepsense_output"
LOG_DIR    = os.path.join(OUTPUT_DIR, "tensorboard_logs",
             datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print("TensorFlow:", tf.__version__)
print("Output dir:", OUTPUT_DIR)

In [ ]:
# Cell 4 - Preprocess
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head(3))

DROP_COLS = ["dataset_source", "participant_id", "stress_score", "occupation"]
df = df.drop(columns=DROP_COLS)

gender_map   = {"Male": 0, "Female": 1, "Other": 2}
agegroup_map = {"13-18": 0, "19-35": 1, "36-59": 2}
df["gender"]    = df["gender"].map(gender_map).fillna(2)
df["age_group"] = df["age_group"].map(agegroup_map).fillna(1)

age_dummies = pd.get_dummies(df["age_group"].map({0:"teen",1:"young_adult",2:"adult"}), prefix="age")
df = pd.concat([df.drop("age_group", axis=1), age_dummies], axis=1)

TARGET       = "stress_risk"
FEATURE_COLS = [c for c in df.columns if c != TARGET]
X = df[FEATURE_COLS].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

print("Features:", FEATURE_COLS)
print("Class: 0={} 1={}".format(int((y==0).sum()), int((y==1).sum())))

neg, pos = (y==0).sum(), (y==1).sum()
class_weight = {0: (neg+pos)/(2*neg), 1: (neg+pos)/(2*pos)}

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15/0.85, random_state=SEED, stratify=y_temp)
print("Train/Val/Test:", len(X_train), "/", len(X_val), "/", len(X_test))

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val   = scaler.transform(X_val).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

SCALER_PATH = os.path.join(OUTPUT_DIR, "scaler_params.json")
META_PATH   = os.path.join(OUTPUT_DIR, "feature_meta.json")
with open(SCALER_PATH, "w") as f:
    json.dump({"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
               "feature_cols": FEATURE_COLS}, f, indent=2)
with open(META_PATH, "w") as f:
    json.dump({"feature_cols": FEATURE_COLS, "gender_map": gender_map,
               "agegroup_map": agegroup_map}, f, indent=2)
print("Preprocessing selesai")

In [ ]:
# Cell 5 - Custom Components (Main Quest)

# ── Custom Layer ──────────────────────────────────────────────
class AttentionScaling(layers.Layer):
    """Custom Layer: Soft attention gate per fitur"""
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.attention_dense = layers.Dense(units, activation="sigmoid", name="attn_gate")
    def call(self, inputs):
        return inputs * self.attention_dense(inputs)
    def get_config(self):
        cfg = super().get_config()
        cfg["units"] = self.units
        return cfg

# ── Custom Loss ───────────────────────────────────────────────
class FocalLoss(keras.losses.Loss):
    """Custom Loss: Focal Loss untuk class imbalance"""
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha
    def call(self, y_true, y_pred):
        y_true  = tf.cast(y_true, tf.float32)
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce     = -y_true*tf.math.log(y_pred) - (1-y_true)*tf.math.log(1-y_pred)
        p_t     = y_true*y_pred + (1-y_true)*(1-y_pred)
        at      = y_true*self.alpha + (1-y_true)*(1-self.alpha)
        return tf.reduce_mean(at * tf.pow(1.0-p_t, self.gamma) * bce)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"gamma": self.gamma, "alpha": self.alpha})
        return cfg

# ── Custom Callback ───────────────────────────────────────────
class EarlyStoppingWithLog(keras.callbacks.Callback):
    """Custom Callback: Early stop + simpan log JSON per epoch"""
    def __init__(self, patience=10, **kwargs):
        super().__init__(**kwargs)
        self.patience     = patience
        self.best_auc     = 0.0
        self.wait         = 0
        self.best_weights = None
        self.history_log  = []
    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        val_auc = logs.get("val_auc", 0)
        self.history_log.append({"epoch": epoch+1,
                                  **{k: round(float(v),5) for k,v in logs.items()}})
        if val_auc > self.best_auc:
            self.best_auc     = val_auc
            self.best_weights = self.model.get_weights()
            self.wait         = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
                self.model.set_weights(self.best_weights)
                print(f"EarlyStopping epoch {epoch+1}, best AUC={self.best_auc:.4f}")
    def on_train_end(self, logs=None):
        log_path = os.path.join(OUTPUT_DIR, "training_log.json")
        with open(log_path, "w") as f:
            json.dump(self.history_log, f, indent=2)
        print("Log disimpan:", log_path)

print("Custom Layer    : AttentionScaling  OK")
print("Custom Loss     : FocalLoss         OK")
print("Custom Callback : EarlyStoppingWithLog OK")

In [ ]:
# Cell 6 - Build Model (Functional API - Main Quest)
n_features = X_train.shape[1]

def build_model(n_features, dropout=0.3):
    inputs = keras.Input(shape=(n_features,), name="features")
    x  = layers.Dense(128, activation="relu",
                       kernel_regularizer=keras.regularizers.l2(1e-4), name="dense_1")(inputs)
    x  = layers.BatchNormalization(name="bn_1")(x)
    x  = AttentionScaling(128, name="attention_1")(x)
    x  = layers.Dropout(dropout, name="drop_1")(x)
    x2 = layers.Dense(64, activation="relu",
                       kernel_regularizer=keras.regularizers.l2(1e-4), name="dense_2")(x)
    x2 = layers.BatchNormalization(name="bn_2")(x2)
    x2 = layers.Dropout(dropout, name="drop_2")(x2)
    x3 = layers.Dense(32, activation="relu", name="dense_3")(x2)
    x3 = layers.BatchNormalization(name="bn_3")(x3)
    out = layers.Dense(1, activation="sigmoid", name="stress_risk")(x3)
    return keras.Model(inputs=inputs, outputs=out, name="SleepSense_StressRisk")

model = build_model(n_features)
model.summary()

In [ ]:
# Cell 7 - Training dengan TensorBoard (Main Quest + Side Quest)
EPOCHS     = 100
BATCH_SIZE = 256

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=FocalLoss(gamma=2.0, alpha=0.35),
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.AUC(name="auc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
    ]
)

callbacks = [
    EarlyStoppingWithLog(patience=12),
    # TensorBoard - Side Quest
    keras.callbacks.TensorBoard(log_dir=LOG_DIR, histogram_freq=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_auc", factor=0.5,
                                      patience=5, mode="max", verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Cell 8 - Custom Training Loop tf.GradientTape (Side Quest)
print("Side Quest: tf.GradientTape Custom Training Loop")
print("=" * 50)

REFINE_EPOCHS = 5
REFINE_LR     = 1e-4
focal_loss_fn = FocalLoss(gamma=2.0, alpha=0.35)
optimizer_gt  = keras.optimizers.Adam(learning_rate=REFINE_LR)
acc_metric    = keras.metrics.BinaryAccuracy()
auc_metric    = keras.metrics.AUC()

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(4096).batch(BATCH_SIZE)
val_ds   = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE)
gt_log   = []

for epoch in range(REFINE_EPOCHS):
    epoch_loss = []
    acc_metric.reset_state()
    auc_metric.reset_state()

    for x_batch, y_batch in train_ds:
        with tf.GradientTape() as tape:
            preds = model(x_batch, training=True)
            loss  = focal_loss_fn(y_batch, tf.squeeze(preds))
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer_gt.apply_gradients(zip(grads, model.trainable_variables))
        epoch_loss.append(float(loss))
        acc_metric.update_state(y_batch, preds)
        auc_metric.update_state(y_batch, preds)

    train_loss = np.mean(epoch_loss)
    train_acc  = float(acc_metric.result())
    train_auc  = float(auc_metric.result())

    acc_metric.reset_state()
    auc_metric.reset_state()
    for x_batch, y_batch in val_ds:
        preds = model(x_batch, training=False)
        acc_metric.update_state(y_batch, preds)
        auc_metric.update_state(y_batch, preds)

    val_acc = float(acc_metric.result())
    val_auc = float(auc_metric.result())

    entry = {"epoch": epoch+1, "loss": round(train_loss,5), "acc": round(train_acc,5),
             "auc": round(train_auc,5), "val_acc": round(val_acc,5), "val_auc": round(val_auc,5)}
    gt_log.append(entry)
    print(f"Refine {epoch+1}/{REFINE_EPOCHS} | loss={train_loss:.4f} acc={train_acc:.4f} auc={train_auc:.4f} | val_acc={val_acc:.4f} val_auc={val_auc:.4f}")

with open(os.path.join(OUTPUT_DIR, "gradienttape_log.json"), "w") as f:
    json.dump(gt_log, f, indent=2)
print("GradientTape refinement selesai")

In [ ]:
# Cell 9 - Evaluasi & Plot (Side Quest: target accuracy >= 85%)
from sklearn.metrics import confusion_matrix, mean_absolute_error

y_pred_prob = model.predict(X_test, verbose=0).flatten()
y_pred      = (y_pred_prob >= 0.5).astype(int)

acc = (y_pred == y_test).mean()
auc = roc_auc_score(y_test, y_pred_prob)
f1  = f1_score(y_test, y_pred)
cm  = confusion_matrix(y_test, y_pred)

print("=" * 50)
print("TEST RESULTS")
print("=" * 50)
print(f"Accuracy : {acc:.4f}  {'PASS' if acc >= 0.85 else 'FAIL'} (target >= 0.85)")
print(f"AUC-ROC  : {auc:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"Confusion Matrix:")
print(cm)
print()
print(classification_report(y_test, y_pred, target_names=["No Risk","At Risk"]))

# Simpan metrics
with open(os.path.join(OUTPUT_DIR, "test_metrics.json"), "w") as f:
    json.dump({"accuracy": round(float(acc),6), "auc_roc": round(float(auc),6),
               "f1_score": round(float(f1),6), "confusion_matrix": cm.tolist()}, f, indent=2)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Val")
axes[0].axhline(0.85, color="red", ls="--", label="Target 85%")
axes[0].set_title("Accuracy"); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(history.history["auc"], label="Train")
axes[1].plot(history.history["val_auc"], label="Val")
axes[1].set_title("AUC-ROC"); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=120)
plt.show()

In [ ]:
# Cell 10 - Simpan Model (Main Quest: .keras + SavedModel)
MODEL_KERAS = os.path.join(OUTPUT_DIR, "sleepsense_model.keras")
MODEL_SM    = os.path.join(OUTPUT_DIR, "sleepsense_savedmodel")

# Format .keras
model.save(MODEL_KERAS)
print("Saved .keras     :", MODEL_KERAS)

# Format SavedModel
model.export(MODEL_SM)
print("Saved SavedModel :", MODEL_SM)

print()
print("Semua file output:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(path):
        print(f"  {f}  ({os.path.getsize(path)//1024} KB)")
    else:
        print(f"  {f}/  (folder)")

In [ ]:
# Cell 11 - Inference Demo (Main Quest)
print("INFERENCE DEMO")
print("Output adalah SCREENING AWAL, bukan diagnosis medis")
print("=" * 50)

def preprocess_input(user_input):
    with open(SCALER_PATH) as f: sp = json.load(f)
    with open(META_PATH) as f:   mt = json.load(f)
    age       = user_input.get("age", 25)
    age_label = "teen" if age <= 18 else ("young_adult" if age <= 35 else "adult")
    row = {
        "age":                             float(age),
        "gender":                          float(mt["gender_map"].get(user_input.get("gender","Male"), 0)),
        "sleep_duration_hours":            float(user_input.get("sleep_duration_hours", 7.0)),
        "sleep_quality_score":             float(user_input.get("sleep_quality_score", 5.0)),
        "daily_screen_time_hours":         float(user_input.get("daily_screen_time_hours", 4.0)),
        "pre_sleep_screen_time_hours":     float(user_input.get("pre_sleep_screen_time_hours", 1.0)),
        "physical_activity_minutes":       float(user_input.get("physical_activity_minutes", 30.0)),
        "caffeine_intake_cups":            float(user_input.get("caffeine_intake_cups", 2.0)),
        "mental_fatigue_score":            float(user_input.get("mental_fatigue_score", 5.0)),
        "notifications_received_per_day":  float(user_input.get("notifications_received_per_day", 50.0)),
        "age_adult":       1.0 if age_label=="adult"       else 0.0,
        "age_teen":        1.0 if age_label=="teen"        else 0.0,
        "age_young_adult": 1.0 if age_label=="young_adult" else 0.0,
    }
    vec   = np.array([row[f] for f in sp["feature_cols"]], dtype=np.float32)
    mean  = np.array(sp["mean"],  dtype=np.float32)
    scale = np.array(sp["scale"], dtype=np.float32)
    return ((vec - mean) / scale).reshape(1, -1)

def interpret(prob):
    if prob < 0.35:   level, label = "Rendah", "No Risk"
    elif prob < 0.65: level, label = "Sedang", "Moderate Risk"
    else:             level, label = "Tinggi", "At Risk"
    return {"risk_label": label, "risk_level": level, "risk_probability": round(float(prob),4)}

CASES = [
    {"label": "Kasus 1 - Risiko Rendah",
     "data": {"age":28,"gender":"Female","sleep_duration_hours":8.0,"sleep_quality_score":8.0,
              "daily_screen_time_hours":3.0,"pre_sleep_screen_time_hours":0.5,
              "physical_activity_minutes":60,"caffeine_intake_cups":1,
              "mental_fatigue_score":3.0,"notifications_received_per_day":30}},
    {"label": "Kasus 2 - Risiko Sedang",
     "data": {"age":22,"gender":"Male","sleep_duration_hours":6.5,"sleep_quality_score":5.0,
              "daily_screen_time_hours":6.0,"pre_sleep_screen_time_hours":1.5,
              "physical_activity_minutes":30,"caffeine_intake_cups":3,
              "mental_fatigue_score":6.0,"notifications_received_per_day":80}},
    {"label": "Kasus 3 - Risiko Tinggi",
     "data": {"age":20,"gender":"Male","sleep_duration_hours":4.5,"sleep_quality_score":3.0,
              "daily_screen_time_hours":10.0,"pre_sleep_screen_time_hours":3.0,
              "physical_activity_minutes":5,"caffeine_intake_cups":5,
              "mental_fatigue_score":9.0,"notifications_received_per_day":200}},
]

for case in CASES:
    x    = preprocess_input(case["data"])
    prob = float(model(x, training=False).numpy().flatten()[0])
    r    = interpret(prob)
    print(f"\n{case['label']}")
    print(f"  Prob  : {r['risk_probability']:.4f}")
    print(f"  Risiko: {r['risk_level']} ({r['risk_label']})")

In [ ]:
# Cell 12 - TensorBoard (Side Quest)
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/tensorboard_logs

In [ ]:
# Cell 13 - Download semua artifact untuk Flask API
import shutil
from google.colab import files

zip_path = "/content/sleepsense_flask_models"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print("Mendownload sleepsense_flask_models.zip ...")
files.download(zip_path + ".zip")
print()
print("Letakkan isi ZIP ke folder models/ di project Flask:")
print("  sleepsense_model.keras       -> models/sleepsense_model.keras")
print("  sleepsense_savedmodel/       -> models/sleepsense_savedmodel/")
print("  scaler_params.json           -> models/scaler_params.json")
print("  feature_meta.json            -> models/feature_meta.json")